*Reincidencia de sentenciados en prisiones del Perú*

***Integrantes:***
* *Renzo Dacillo*
* *Pierre Lavergne*
* *Luis Alberto Carrasco*

# **Modelos implementados**


## 1. Preparación
…

#### 1.1 Librairies y importaciones

In [ ]:
# Path

import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Librairies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Funcciones utiles

import src.data_loader as loader
import src.models as models
import src.evaluation as eval

ModuleNotFoundError: No module named 'tensorflow'

#### 1.2 Carga de datos y modelos

In [9]:
X_train, X_test, y_train, y_test = loader.prepararDatos()
X_train_scaled, X_test_scaled = loader.escalarDatos(X_train, X_test)

pd.DataFrame({
    "Conjunto": ["Entrenamiento (Train)", "Prueba (Test)", "Total"],
    "Muestras": [len(X_train), len(X_test), len(X_train) + len(X_test)],
    "Features": [X_train.shape[1], X_test.shape[1], X_train.shape[1]],
    "Proporción Reincidentes": [y_train.mean(), y_test.mean(), pd.concat([y_train, y_test]).mean()]
})

--- Cargando dataset version 04 ---
Dataset cargado: 36989 filas x 133 columnas
Datos separados (test_size=0.2): Train 29591 filas | Test 7398 filas
Nombre de NaN restantes en el cojunto : 0
Escalado (StandardScaler) aplicado a: ['EDAD', 'EDAD_SIN_MAMA', 'EDAD_SIN_PAPA']


,Conjunto,Muestras,Features,Proporción Reincidentes
0,Entrenamiento (Train),29591,132,0.243250
1,Prueba (Test),7398,132,0.243174
2,Total,36989,132,0.243234


# 2. Optimización de hyperparametros
…

### 2.1 Logistic Regresion (Basline)

In [ ]:
param_dist_logistic = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0],
    "penalty": ["l2"],
    "solver": ["lbfgs", "saga"],
}

res_logistic = models.optimizarModelo(
    "LogisticRegression",
    param_dist_logistic,
    X_train,
    X_train_scaled,
    y_train,
    n_iter=10
)

### 2.2 KNN (K-Nearest Neighbors)

In [ ]:
param_dist_knn = {
    "n_neighbors": [3, 5, 7, 11, 15, 21],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
    "p": [1, 2],
}

res_knn = models.optimizarModelo(
    "KNN",
    param_dist_knn,
    X_train,
    X_train_scaled,
    y_train,
    n_iter=10
)

### 2.3 Random Forest

In [ ]:
param_dist_forest = {
    "n_estimators": [200, 300, 400],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [2, 4, 8],
    "max_features": ["sqrt", "log2"],
}

res_forest = models.optimizarModelo(
    "RandomForest",
    param_dist_forest,
    X_train,
    X_train_scaled,
    y_train,
    n_iter=15
)

### 2.4 MLP (Multi-Layer-Perceptron)

In [ ]:
param_dist_mlp = {
    "hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01, 0.1],
    "learning_rate": ["constant", "adaptive"],
    "learning_rate_init": [0.0005, 0.001, 0.005],
    "batch_size": ["auto", 32, 64],
}

res_mlp = models.optimizarModelo(
    "MLPClassifier",
    param_dist_mlp,
    X_train,
    X_train_scaled,
    y_train,
    n_iter=15
)

# 3. Prueba de los modelos
…

### 3.1 Resumen de modelos

In [ ]:
modelos_entrenados = {
    "Logistic Regression": res_logistic,
    "KNN": res_knn,
    "Random Forest": res_forest,
    "MLP (Red Neuronal)": res_mlp,
}

resumen_cv = pd.DataFrame([
    {
        "Modelo": nombre,
        "Mejor ROC-AUC (CV)": round(info["best_cv_auc"], 4),
        "Requiere Escalado": info["requiere_escalado"],
        "Mejores Hiperparámetros": info["best_params"],
    }
    for nombre, info in modelos_entrenados.items()
]).sort_values(by="Mejor ROC-AUC (CV)", ascending=False)

display(resumen_cv)

### 3.2 Tests

In [ ]:
df_resultados, predicciones = eval.compararModelos(
    modelos_entrenados,
    X_test,
    X_test_scaled,
    y_test
)

display(df_resultados.round(4))

### 3.3 General evaluación

In [ ]:
eval.graficarMatricesConfusion(predicciones, y_test)
eval.graficarCurvasROC_PR(predicciones, y_test)

# 4. Evaluación especifica
…

In [ ]:
mejor_nombre = df_resultados.index[0]
mejor_info = modelos_entrenados[mejor_nombre]
print(f"Mejor modelo seleccionado: {mejor_nombre} (ROC-AUC Test = {df_resultados.loc[mejor_nombre, 'ROC-AUC']:.4f})\n")

eval.imprimirReporteDetallado(
    mejor_nombre,
    y_test,
    predicciones[mejor_nombre]["y_pred"]
)

df_importancias = eval.graficarImportanciaVariables(
    mejor_info,
    feature_names=X_train.columns,
    nombre_modelo=mejor_nombre,
    top_n=20
)

# MLP o KNN no tienen feature_importances_
# Mostramos la de Random Forest si el mejor modelo es MLP o KNN
if df_importancias is None in modelos_entrenados:
    print("\nImportancia de variables basada en Random Forest :")
    eval.graficarImportanciaVariables(
        modelos_entrenados["Random Forest"],
        feature_names=X_train.columns,
        nombre_modelo="Random Forest",
        top_n=20
    )

# Curva de aprendizaje de la Red Neuronal en caso
if mejor_nombre = "MLP (Red Neuronal)":
    mlp_model = modelos_entrenados["MLP (Red Neuronal)"]["model"]
    plt.figure(figsize=(8, 4))
    plt.plot(mlp_model.loss_curve_, label="Train Loss")
    plt.title("Curva de Aprendizaje (Loss) - MLPClassifier")
    plt.xlabel("Iteraciones (Épocas)")
    plt.ylabel("Log-Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()